In [7]:
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
openai_client = OpenAI()

In [8]:
def llm(prompt):
    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=prompt
    )
    return response.output_text

In [9]:
llm("Hey, what's up?")

'Hey! Not much — I’m here and ready to help. What’s on your mind?'

In [10]:
question = "I just discovered the course. Can I join now?"
answer = llm(question)
print(answer)

Maybe — it depends on the course and whether it’s still open for enrollment.

If you want, I can help you figure it out quickly. Usually the key questions are:
- Is enrollment still open?
- Has the course already started?
- Is there a waiting list or late-entry option?
- Do you need to catch up on missed material?

If you’re asking about a specific course, send me:
- the course name
- the platform or school
- and any deadlines you see

and I’ll help you determine whether you can still join and what to do next.


In [11]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

In [12]:
answer = llm(prompt)
print(answer)

Yes, you can still join now. If you want to receive a certificate, make sure to submit your project while submissions are still being accepted.


In [13]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

In [14]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1406

In [15]:
documents[0]

{'id': '0e38656cfb',
 'course': 'machine-learning-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'How do I submit homework?',
 'answer': "- Do the tasks locally\n- Publish your code (e.g., in your own GitHub repo)\n- Submit your answers via the homework form and include the URL to your code\n- You will see the answers only after the deadline\n- Homeworks are in the cohorts folder, e.g. for 2025 it's [`cohorts/2025`](https://github.com/DataTalksClub/machine-learning-zoomcamp/tree/master/cohorts/2025)\n- The forms for submitting the homework are in the [course management platform](https://courses.datatalks.club/)"}

In [16]:
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

In [17]:
question = "I just discovered the course. Can I join now?"

search_results = index.search(
    question,
    boost_dict={"question": 2.0, "section": 0.5},
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '5cc511f85b',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Does the course certificate show the number of course hours?',
  'answer': 'No. The certificate does not 

In [18]:
[doc["question"] for doc in search_results]

['I just discovered the course. Can I still join?',
 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
 'Does the course certificate show the number of course hours?',
 "Why do we need orchestration / Kestra — can't I just run the code in a notebook?",
 'Certificate: Can I follow the course in a self-paced mode and get a certificate?']

In [19]:
results = index.search(
    question,
    num_results=5,
    boost_dict={"question": 2.0, "section": 0.5}
)

In [20]:
[doc["question"] for doc in results]

['I just discovered the course. Can I still join?',
 'The course has already started. Can I still join it?',
 'Course - Can I still join the course after the start date?',
 'Course: Can I still join the course after the start date?',
 'Course: When does the course start?']

In [21]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

In [22]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

In [23]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

In [24]:
prompt = build_prompt(question, search_results)

print(prompt)

Question:
I just discovered the course. Can I join now?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Does the course certificate show the number of course hours?
A: No. The certificate does not state a total number of hours.

Module 3: Orchestration
Q: Why do we need orchestration / Kestra — can't I just run the code in a notebook?
A: Notebooks are great for learning and experimenting, but real AI w

In [25]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=prompt
)

In [26]:
response.output[0]

ResponseOutputMessage(id='msg_00088c325062bc65006a9953e1de6887d0886da2fc8be23a36', content=[ResponseOutputText(annotations=[], text='Yes — you can join now.  \n\nIf you want to receive a certificate, make sure to submit your project while the course is still accepting submissions.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')

In [27]:
response.output[0].content[0].text

'Yes — you can join now.  \n\nIf you want to receive a certificate, make sure to submit your project while the course is still accepting submissions.'

In [28]:
response.output_text

'Yes — you can join now.  \n\nIf you want to receive a certificate, make sure to submit your project while the course is still accepting submissions.'

In [29]:
response.usage

ResponseUsage(input_tokens=506, input_tokens_details=InputTokensDetails(cache_write_tokens=0, cached_tokens=0), output_tokens=34, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=540)

In [30]:
input_price = 0.75 / 1_000_000
output_price = 4.50 / 1_000_000

cost = (
    response.usage.input_tokens * input_price +
    response.usage.output_tokens * output_price
)

cost

0.0005325

In [31]:
message_history = [
    {"role": "developer", "content": INSTRUCTIONS},
    {"role": "user", "content": prompt}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=message_history
)

In [32]:
def search(question, course="llm-zoomcamp"):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

def llm(instructions, user_prompt, model="gpt-5.4-mini"):
    message_history = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.create(
        model=model,
        input=message_history
    )

    return response.output_text

def rag(query, model="gpt-5.4-mini"):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer

In [33]:
answer = rag("I just discovered the course. Can I join now?")
print(answer)

Yes, you can still join now. If you want a certificate, you need to submit your project while submissions are still open.


In [34]:
rag("How do I get a certificate?")

'You can get a certificate only if you finish the course with a live cohort.\n\nTo qualify, you need to:\n- complete the capstone project, and\n- finish the required peer reviews.\n\nHomework is not mandatory for the certificate. You can study and prepare in self-paced mode, but project submission and peer review must happen while a live cohort is accepting them.'

In [35]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
)

response.output_text

'Maybe — it depends on the course’s enrollment rules and whether registration is still open.\n\nIf you want, I can help you figure it out. Please send me:\n- the course name\n- the school/platform\n- whether it’s live, self-paced, or part of a semester\n\nThen I can help you draft a message to the instructor or check what to look for.'

In [36]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [37]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [38]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output

[ResponseFunctionToolCall(arguments='{"query":"Can I join if I just discovered the course? enrollment after start late join join the course discovered course"}', call_id='call_1o4YP16nrFUAAuJYPfCE0Lh0', name='search', type='function_call', id='fc_0cfe5233433d4099006a9953e849f487d088f620bd0d408bfb', caller=None, namespace=None, status='completed')]

In [41]:
import json

call = response.output[0]
args = json.loads(call.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)

In [40]:
messages.extend(response.output)

messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})

In [42]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output_text

'Yes — you can still join and start learning anytime.\n\nIf your goal is a certificate, make sure you submit your project while submissions are still open.'

In [43]:
usage = response.usage
usage.input_tokens, usage.output_tokens

(793, 34)

In [44]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 0.15
    OUTPUT_PRICE_PER_MILLION = 0.60

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

result = calculate_gpt54mini_price(652, 33)
print("Total cost: $", round(result["total_cost"], 8))

Total cost: $ 0.0001176


In [45]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [46]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [47]:
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

messages.extend(response.output)
has_function_calls = False

for item in response.output:
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
        has_function_calls = True

    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)

function_call: search {"query":"join course discovered course can I join enrollment late registration FAQ"}


In [48]:
it = 1

while True:
    print(f"iteration #{it}...")
    has_function_calls = False

    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool],
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == "function_call":
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == "message":
            print("ASSISTANT:")
            print(item.content[0].text)

    it = it + 1
    if has_function_calls == False:
        break

iteration #1...
ASSISTANT:
Yes — you can still join.

According to the course FAQ, if you just discovered the course, you’re welcome to start learning and participating. If you want a certificate, you’ll need to submit your project while submissions are still open.

If you want, I can also help clarify how registration, homework, and certificates work.


In [49]:
def agent_loop(instructions, question, model="gpt-5.4-mini") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

In [50]:
agent_loop(instructions, "How do I run Olama locally?")

iteration #1...
function_call: search {"query":"Ollama local install run locally course FAQ"}
function_call: search {"query":"Olama local run install model serve FAQ"}
function_call: search {"query":"Ollama localhost run local FAQ"}
iteration #2...
ASSISTANT:
To run Ollama locally:

1. **Install Ollama**
   - macOS: download and install from https://ollama.com/download
   - Windows: download the `.msi` installer from the same page
   - Linux:
     ```bash
     curl -fsSL https://ollama.com/install.sh | sh
     ```

2. **Start a model locally**
   ```bash
   ollama run llama3
   ```
   This downloads the model and opens a local chat interface.

3. **Check that the local server is running**
   ```bash
   curl http://localhost:11434
   ```
   You should get a response from Ollama.

4. **Use it from Python**
   ```bash
   pip install ollama
   ```
   Example:
   ```python
   import ollama

   response = ollama.chat(
       model='llama3',
       messages=[{"role": "user", "content": "Hello

'To run Ollama locally:\n\n1. **Install Ollama**\n   - macOS: download and install from https://ollama.com/download\n   - Windows: download the `.msi` installer from the same page\n   - Linux:\n     ```bash\n     curl -fsSL https://ollama.com/install.sh | sh\n     ```\n\n2. **Start a model locally**\n   ```bash\n   ollama run llama3\n   ```\n   This downloads the model and opens a local chat interface.\n\n3. **Check that the local server is running**\n   ```bash\n   curl http://localhost:11434\n   ```\n   You should get a response from Ollama.\n\n4. **Use it from Python**\n   ```bash\n   pip install ollama\n   ```\n   Example:\n   ```python\n   import ollama\n\n   response = ollama.chat(\n       model=\'llama3\',\n       messages=[{"role": "user", "content": "Hello!"}]\n   )\n\n   print(response[\'message\'][\'content\'])\n   ```\n\nIf you get a “connection refused” error, restart the server with:\n```bash\nollama serve\n```\nor, in a notebook:\n```bash\n!nohup ollama serve > nohup.out

In [51]:
agent_loop(instructions, "I just discovered the course. Can I still join it?")

iteration #1...
function_call: search {"query":"join course late discovered course can I still join"}
iteration #2...
ASSISTANT:
Yes — you can still join the course.

If you want a certificate, make sure you submit your project while submissions are still being accepted. Homework is not required for the certificate, but the capstone/project and required peer reviews must be completed during a live cohort.

If you'd like, I can also help clarify how the certificate requirements work. Are there other areas you want to explore?


"Yes — you can still join the course.\n\nIf you want a certificate, make sure you submit your project while submissions are still being accepted. Homework is not required for the certificate, but the capstone/project and required peer reviews must be completed during a live cohort.\n\nIf you'd like, I can also help clarify how the certificate requirements work. Are there other areas you want to explore?"

In [52]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "I just discovered the course. Can I join it?")

iteration #1...
function_call: search {"query":"join course discovered course can I join enrollment late registration audit"}
iteration #2...
ASSISTANT:
Yes — you can still join and start learning.

If you want a certificate, though, you’ll need to submit your project while the course is still accepting submissions.

Would you like me to also explain how registration and homework submission work?


'Yes — you can still join and start learning.\n\nIf you want a certificate, though, you’ll need to submit your project while the course is still accepting submissions.\n\nWould you like me to also explain how registration and homework submission work?'

In [53]:
agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit chess opening queen gambit"}
iteration #2...
function_call: search {"query":"queen's gambit chess opening definition"}
iteration #3...
ASSISTANT:
The **Queen’s Gambit** is a chess opening that starts with:

1. **d4 d5**
2. **c4**

White offers the **c-pawn** as a “gambit” to distract Black’s center pawn and gain better control of the board, especially the center.

A common continuation is:
- **1. d4 d5 2. c4**

If Black accepts:
- **Queen’s Gambit Accepted**: `1.d4 d5 2.c4 dxc4`

If Black declines:
- **Queen’s Gambit Declined**: `1.d4 d5 2.c4 e6` or similar

It’s one of the most famous and important openings in chess.

If you want, I can also explain:
- how to play it,
- the difference between Accepted and Declined,
- or show a few common traps and ideas.


'The **Queen’s Gambit** is a chess opening that starts with:\n\n1. **d4 d5**\n2. **c4**\n\nWhite offers the **c-pawn** as a “gambit” to distract Black’s center pawn and gain better control of the board, especially the center.\n\nA common continuation is:\n- **1. d4 d5 2. c4**\n\nIf Black accepts:\n- **Queen’s Gambit Accepted**: `1.d4 d5 2.c4 dxc4`\n\nIf Black declines:\n- **Queen’s Gambit Declined**: `1.d4 d5 2.c4 e6` or similar\n\nIt’s one of the most famous and important openings in chess.\n\nIf you want, I can also explain:\n- how to play it,\n- the difference between Accepted and Declined,\n- or show a few common traps and ideas.'

In [54]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit"}
iteration #2...
function_call: search {"query":"queen's gambit chess opening"}
iteration #3...
function_call: search {"query":"queen gambit course FAQ"}
iteration #4...
ASSISTANT:
I couldn’t find a course FAQ entry for “queen gambit,” so it looks like this is off-topic or not covered in the course materials.

If you meant the chess opening, the course FAQ doesn’t have information about it. If you meant something else course-related, feel free to rephrase and I can check again. Is there another area you want to explore?


'I couldn’t find a course FAQ entry for “queen gambit,” so it looks like this is off-topic or not covered in the course materials.\n\nIf you meant the chess opening, the course FAQ doesn’t have information about it. If you meant something else course-related, feel free to rephrase and I can check again. Is there another area you want to explore?'

In [55]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [56]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

In [57]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"}
    )

In [58]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [59]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [60]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [61]:
result = runner.loop(
    prompt="How do I run Olama locally?",
    callback=callback,
)

-> Response received


-> Response received


-> Response received


In [62]:
result.cost

CostInfo(input_cost=Decimal('0.00318675'), output_cost=Decimal('0.0015345'), total_cost=Decimal('0.00472125'))

In [63]:
result.all_messages

[EasyInputMessage(content="You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches. First perform search, analyze the results \nand then perform more searches. \n\nThe question has to be about the course or its logistics, offtopic questions \nshouldn't be answered. If the search returns nothing, it's likely an off-topic question.\nIf you can't answer the question using FAQ, don't do it yourself. Only use the \nfacts from the FAQ database.\n\nAt the end, ask if there are other areas that the user wants to explore.", role='developer', phase=None, type=None),
 EasyInputMessage(content='How do I run Olama locally?', role='user', phase=None, type=None),
 ResponseFunctionToolCall(arguments='{"query":"Olama locally run Ollama local installation course FAQ"}', cal

In [64]:
runner.run()

You: what is the purpose of the rag course


-> Response received


-> Response received


-> Response received


You: How popular is the course?


...

-> Response received


-> Response received


You: stop


Chat ended.


LoopResult(new_messages=[EasyInputMessage(content="You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches. First perform search, analyze the results \nand then perform more searches. \n\nThe question has to be about the course or its logistics, offtopic questions \nshouldn't be answered. If the search returns nothing, it's likely an off-topic question.\nIf you can't answer the question using FAQ, don't do it yourself. Only use the \nfacts from the FAQ database.\n\nAt the end, ask if there are other areas that the user wants to explore.", role='developer', phase=None, type=None), EasyInputMessage(content='what is the purpose of the rag course', role='user', phase=None, type=None), ResponseFunctionToolCall(arguments='{"query":"purpose of the RAG course"}', 